In [1]:
"""
Entrena y evalua los tres modelos (MLP, LSTM, GRU) con validacion cruzada
por paciente, igual que el TFG. Guarda resumen, matrices de confusion y
predicciones por paciente en OUT_DIR.

Balanceo por defecto: class_weight. Se probo SMOTE-Tomek pero contaminaba el
15 % de validacion interna del early stopping con muestras sinteticas, asi que
se cambio a class_weight (penaliza la perdida sin generar muestras). El codigo
de SMOTE-Tomek se deja por si se quiere reactivar separando antes la validacion.

En las secuencias, el padding (ceros con que se completa cada ventana hasta
500 frames) se marca con PADDING_VALUE tras normalizar, para que la capa
Masking de Keras lo ignore.
"""

import os
os.environ["PYTHONHASHSEED"] = "42"

import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

# --- Rutas ---
CSV_FEATURES   = r"C:\Users\josem\Desktop\tfg\dataset_60_topado.csv"
NPZ_SECUENCIAS = r"C:\Users\josem\Desktop\tfg\mfcc_secuencia_60.npz"
OUT_DIR        = r"C:\Users\josem\Desktop\tfg\resultados"

# Balanceo intra-fold: "class_weight" o "smote_tomek".
BALANCEO = "class_weight"

EPOCHS     = 100
BATCH_SIZE = 32
PATIENCE   = 12
VAL_SPLIT  = 0.15
SEED       = 42

CLASES = ["NORMAL", "ASTHMA", "BRON", "COPD", "HEART FAILURE", "PNEUMONIA"]
N_CLASES = len(CLASES)
N_FOLDS = 10

# Valor con el que se marca el padding tras normalizar, para el Masking.
PADDING_VALUE = -99.0


def fijar_semillas(s=SEED):
    random.seed(s)
    np.random.seed(s)
    tf.random.set_seed(s)


# --- Carga de datos ---

def cargar_features():
    # X (956, 41), y entero, fold, pid.
    df = pd.read_csv(CSV_FEATURES, sep=";", decimal=",")
    control = ["archivo", "diagnostico", "ventana", "pid", "fold"]
    feats = [c for c in df.columns if c not in control]
    assert len(feats) == 41, f"Se esperaban 41 features, hay {len(feats)}"
    clase_a_int = {c: i for i, c in enumerate(CLASES)}
    X = df[feats].to_numpy(dtype=np.float32)
    y = df["diagnostico"].map(clase_a_int).to_numpy(dtype=np.int64)
    fold = df["fold"].to_numpy(dtype=np.int64)
    pid = df["pid"].to_numpy()
    return X, y, fold, pid


def cargar_secuencias():
    # X (956, 500, 13), y, fold, pid y mascara (True = frame valido).
    d = np.load(NPZ_SECUENCIAS, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y_int"].astype(np.int64)
    fold = d["fold"].astype(np.int64)
    pid = d["pid"]
    clases_npz = [str(c) for c in d["clases"]]
    assert clases_npz == CLASES, f"Orden de clases distinto: {clases_npz}"
    # Un frame es padding si sus 13 MFCC son todos cero.
    mascara_valida = ~np.all(X == 0.0, axis=2)   # (956, 500)
    return X, y, fold, pid, mascara_valida


# --- Balanceo intra-fold ---

def aplicar_smote_tomek(X, y):
    # SMOTE-Tomek sobre el train. Aplana las secuencias a 2D y vuelve a 3D.
    from imblearn.combine import SMOTETomek
    from imblearn.over_sampling import SMOTE

    forma = X.shape
    X2 = X.reshape(len(X), -1) if X.ndim == 3 else X
    min_clase = np.bincount(y).min()
    k = max(1, min(5, int(min_clase) - 1))
    st = SMOTETomek(smote=SMOTE(k_neighbors=k, random_state=SEED),
                    random_state=SEED)
    Xr, yr = st.fit_resample(X2, y)
    if len(forma) == 3:
        Xr = Xr.reshape(-1, forma[1], forma[2])
    return Xr.astype(np.float32), yr.astype(np.int64)


def pesos_clase(y):
    # {clase: peso} para el class_weight de Keras.
    pesos = compute_class_weight("balanced", classes=np.unique(y), y=y)
    mapa = {c: 1.0 for c in range(N_CLASES)}
    for c, w in zip(np.unique(y), pesos):
        mapa[int(c)] = float(w)
    return mapa


# --- Normalizacion intra-fold (ajustada solo sobre el train) ---

def normaliza_features(Xtr, Xval):
    sc = MinMaxScaler().fit(Xtr)
    return (sc.transform(Xtr).astype(np.float32),
            sc.transform(Xval).astype(np.float32))


def normaliza_secuencias(Xtr, Xval, mask_tr, mask_val):
    # z-score por coeficiente, ajustado solo sobre los frames validos del
    # train. El padding se rellena con PADDING_VALUE para que el Masking
    # lo ignore (si no, quedaria en -media/std y no seria detectable).
    n_tr, t, c = Xtr.shape
    n_val = Xval.shape[0]

    sc = StandardScaler().fit(Xtr[mask_tr])
    Xtr_n  = sc.transform(Xtr.reshape(-1, c)).reshape(n_tr, t, c).astype(np.float32)
    Xval_n = sc.transform(Xval.reshape(-1, c)).reshape(n_val, t, c).astype(np.float32)

    Xtr_n[~mask_tr]   = PADDING_VALUE
    Xval_n[~mask_val] = PADDING_VALUE
    return Xtr_n, Xval_n


# --- Metricas (one-vs-rest macro, como el TFG) ---

def metricas(y_true, y_pred):
    # Para cada clase se trata como binario (clase vs resto) y se promedia.
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASES)))
    total = cm.sum()
    accs, sens, specs, f1s = [], [], [], []
    for c in range(N_CLASES):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = total - tp - fn - fp
        acc_c  = (tp + tn) / total if total > 0 else 0.0
        sen_c  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spe_c  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        prec_c = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1_c   = (2 * prec_c * sen_c / (prec_c + sen_c)) if (prec_c + sen_c) > 0 else 0.0
        accs.append(acc_c)
        sens.append(sen_c)
        specs.append(spe_c)
        f1s.append(f1_c)
    return {
        "accuracy":      float(np.mean(accs)),
        "sensibilidad":  float(np.mean(sens)),
        "especificidad": float(np.mean(specs)),
        "f1":            float(np.mean(f1s)),
        "kappa":         float(cohen_kappa_score(y_true, y_pred)),
    }


# --- Modelos ---

def build_mlp(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(N_CLASES, activation="softmax"),
    ])
    m.compile(optimizer=optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def build_lstm(timesteps, n_feat):
    m = models.Sequential([
        layers.Input(shape=(timesteps, n_feat)),
        layers.Masking(mask_value=PADDING_VALUE),
        layers.LSTM(64, recurrent_dropout=0.2),
        layers.Dropout(0.4),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(N_CLASES, activation="softmax"),
    ])
    m.compile(optimizer=optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def build_gru(timesteps, n_feat):
    m = models.Sequential([
        layers.Input(shape=(timesteps, n_feat)),
        layers.Masking(mask_value=PADDING_VALUE),
        layers.GRU(64, recurrent_dropout=0.2),
        layers.Dropout(0.4),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(N_CLASES, activation="softmax"),
    ])
    m.compile(optimizer=optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


# --- Validacion cruzada por paciente ---

def evaluar(nombre, X, y, fold, pid, tipo, build_fn, mascara=None):
    # Entrena sobre ventanas, predice ventanas y agrega a paciente por voto
    # mayoritario. Las metricas se calculan una vez sobre los 60 pacientes.
    pac_true, pac_pred, pac_id = [], [], []

    for k in range(1, N_FOLDS + 1):
        tr = fold != k
        te = fold == k
        Xtr, ytr = X[tr], y[tr]
        Xte, yte = X[te], y[te]
        pid_te = pid[te]

        if tipo == "features":
            Xtr, Xte = normaliza_features(Xtr, Xte)
        else:
            Xtr, Xte = normaliza_secuencias(Xtr, Xte, mascara[tr], mascara[te])

        class_weight = None
        if BALANCEO == "smote_tomek":
            Xtr, ytr = aplicar_smote_tomek(Xtr, ytr)
        elif BALANCEO == "class_weight":
            class_weight = pesos_clase(ytr)
        else:
            raise ValueError(f"BALANCEO no valido: {BALANCEO}")

        # Barajar antes del fit (el 15 % del early stopping coge el final).
        rng = np.random.default_rng(SEED + k)
        perm = rng.permutation(len(Xtr))
        Xtr, ytr = Xtr[perm], ytr[perm]

        modelo = build_fn()
        es = callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                     restore_best_weights=True)
        modelo.fit(Xtr, ytr, validation_split=VAL_SPLIT, epochs=EPOCHS,
                   batch_size=BATCH_SIZE, class_weight=class_weight,
                   callbacks=[es], verbose=0)

        pred_win = np.argmax(modelo.predict(Xte, verbose=0), axis=1)
        for p in pd.unique(pid_te):
            mask = pid_te == p
            voto = np.bincount(pred_win[mask], minlength=N_CLASES).argmax()
            pac_true.append(int(yte[mask][0]))
            pac_pred.append(int(voto))
            pac_id.append(p)

        print(f"  [{nombre}] fold {k:2d}/{N_FOLDS}  "
              f"acc_ventana={accuracy_score(yte, pred_win):.3f}  "
              f"pacientes={len(pd.unique(pid_te))}")

        tf.keras.backend.clear_session()

    pac_true = np.array(pac_true)
    pac_pred = np.array(pac_pred)

    m = metricas(pac_true, pac_pred)
    m["accuracy_multiclase"] = float(accuracy_score(pac_true, pac_pred))
    m["modelo"] = nombre
    m["n_pacientes"] = int(len(pac_true))
    cm = confusion_matrix(pac_true, pac_pred, labels=list(range(N_CLASES)))
    detalle = pd.DataFrame({"modelo": nombre, "pid": pac_id,
                            "real": pac_true, "predicho": pac_pred})
    return m, cm, detalle


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    fijar_semillas()

    print(f"Balanceo: {BALANCEO}\n")

    Xf, yf, foldf, pidf = cargar_features()
    Xs, ys, folds, pids, mask_s = cargar_secuencias()

    # Comprobar que CSV y .npz estan alineados.
    assert np.array_equal(foldf, folds), "Los folds del CSV y del .npz no coinciden"
    assert np.array_equal(yf, ys), "Las etiquetas del CSV y del .npz no coinciden"
    assert np.array_equal(pidf.astype(str), pids.astype(str)), \
        "Los pacientes del CSV y del .npz no coinciden"

    config = [
        ("MLP",  Xf, yf, foldf, pidf, "features",  lambda: build_mlp(Xf.shape[1]),               None),
        ("LSTM", Xs, ys, folds, pids, "secuencia", lambda: build_lstm(Xs.shape[1], Xs.shape[2]), mask_s),
        ("GRU",  Xs, ys, folds, pids, "secuencia", lambda: build_gru(Xs.shape[1], Xs.shape[2]),  mask_s),
    ]

    resumenes, detalles = [], []
    for nombre, X, y, fold, pid, tipo, build_fn, mascara in config:
        print(f"=== {nombre} ===")
        m, cm, detalle = evaluar(nombre, X, y, fold, pid, tipo, build_fn, mascara)
        resumenes.append(m)
        detalles.append(detalle)
        pd.DataFrame(cm, index=CLASES, columns=CLASES).to_csv(
            os.path.join(OUT_DIR, f"matriz_confusion_{nombre}.csv"))
        print()

    pd.concat(detalles, ignore_index=True).to_csv(
        os.path.join(OUT_DIR, "predicciones_pacientes.csv"), index=False)
    cols = ["modelo", "accuracy", "sensibilidad", "especificidad", "f1",
            "kappa", "accuracy_multiclase", "n_pacientes"]
    pd.DataFrame(resumenes)[cols].to_csv(
        os.path.join(OUT_DIR, "resumen_modelos.csv"), index=False)

    print("=== RESUMEN (one-vs-rest macro) ===")
    for r in resumenes:
        print(f"\n{r['modelo']}")
        for c in ["accuracy", "sensibilidad", "especificidad", "f1", "kappa"]:
            print(f"  {c:14s}: {r[c]*100:5.2f} %")
        print(f"  (acc. multiclase: {r['accuracy_multiclase']*100:5.2f} %)")
    print(f"\nResultados en: {OUT_DIR}")


if __name__ == "__main__":
    main()

GPU detectada: False (0 dispositivos)
Balanceo: class_weight

Padding en secuencias: media de frames validos por ventana = 500.0 (max 500); minimo = 500

=== MLP ===
  [MLP] fold  1/10  acc_ventana=0.519  pacientes=6

  [MLP] fold  2/10  acc_ventana=0.744  pacientes=6
  [MLP] fold  3/10  acc_ventana=0.692  pacientes=6
  [MLP] fold  4/10  acc_ventana=0.479  pacientes=6
  [MLP] fold  5/10  acc_ventana=0.723  pacientes=6
  [MLP] fold  6/10  acc_ventana=0.663  pacientes=6
  [MLP] fold  7/10  acc_ventana=0.639  pacientes=6
  [MLP] fold  8/10  acc_ventana=0.735  pacientes=6
  [MLP] fold  9/10  acc_ventana=0.639  pacientes=6
  [MLP] fold 10/10  acc_ventana=0.595  pacientes=6

=== LSTM ===
  [LSTM] fold  1/10  acc_ventana=0.647  pacientes=6
  [LSTM] fold  2/10  acc_ventana=0.564  pacientes=6
  [LSTM] fold  3/10  acc_ventana=0.523  pacientes=6
  [LSTM] fold  4/10  acc_ventana=0.448  pacientes=6
  [LSTM] fold  5/10  acc_ventana=0.585  pacientes=6
  [LSTM] fold  6/10  acc_ventana=0.593  pacientes